# Project 2 · Step 4: recurrent encoder (Kazik)

Pipeline for step 4: data, pretrained embeddings, tokenisation, padded batches, the shared folds, a training loop, an LSTM skeleton (section 7, **yours to extend**), a hyperparameter grid, an overfitting check and the single final test evaluation.

Everything uses the team's shared protocol, so the numbers compare directly with the baselines in `shared/results/`:
- `emb-ft-crawl-mean/logreg`: mean-pooled fastText + logistic regression, **28.5%** balanced accuracy;
- `emb-ft-crawl-mean-promptcentred/logreg`: the same with each prompt's mean subtracted, **40.6%**.

Chance is 5.6% (18 classes).

## 1. Setup

In [ ]:
import copy
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "part2" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence
from torch.utils.data import DataLoader, Dataset

from part2 import wordvecs as WV
from shared import eval_harness as H
from shared.data import load_data

DEVICE = "cpu"  # "mps" also works on a Mac, but CPU runs are reproducible and LSTMs this small aren't much faster on MPS
H.set_seed()

## 2. Data

`D.train` is our training split (2,159 passages). `D.test` is the held-out split: **don't touch it** until the final evaluation. Labels are integers `0..17` and `CLASSES[i]` is the model name. `groups` is the prompt id, which the folds use so that no prompt appears on both sides of a split.

In [ ]:
D = load_data()
texts = D.train["text"].tolist()
y = D.y  # np.ndarray of ints, aligned with texts
groups = D.groups  # prompt id per text
CLASSES = D.classes
N_CLASSES = len(CLASSES)

print(
    f"train {len(texts)} texts, test (held out) {len(D.test)}, {N_CLASSES} classes, {len(set(groups))} prompts"
)
pd.Series(y).map(dict(enumerate(CLASSES))).value_counts().sort_index().to_frame("n").T

## 3. Embeddings

`WV.load_subset(name)` returns `(tok, emb)`:
- `tok` is the tokeniser that matches that embedding's vocabulary. It is cased, keeps punctuation as tokens, and splits `don't` the way the embedding was trained.
- `emb.vectors` is a `(V, 300)` numpy array. **Row 0 is `<pad>`** (all zeros) and **row 1 is `<unk>`**. The table holds only the words that occur in our texts (train and Kaggle test), which keeps it small.

Useful for later experiments:
- `emb.random_like(seed)` gives the same vocabulary with random vectors (pretrained vs random initialisation);
- `emb.pca(k)` gives the same vocabulary in k dimensions.

The main embedding is `ft-crawl` (fastText Common Crawl). The others load only if their subset has been built (`python -m part2.prepare_embeddings <name>`).

**`emb_matrix(name)` standardises each of the 300 dimensions** (mean 0, std 1 over the vocabulary; the `<pad>` row stays 0), the same scaling the logistic-regression baseline applies with `StandardScaler`. The raw vectors are tiny, and on them the models barely train: on the same 5 folds, an LSTM (100 units, 1 layer) went from about 16% to 20.5% balanced accuracy with standardisation. `emb_matrix(name, standardise=False)` gives the raw vectors.

In [ ]:
PAD_IDX, UNK_IDX = 0, 1

EMB = {}
for name in ["ft-crawl", "glove-840b", "ft-wiki", "w2v-gnews"]:
    try:
        EMB[name] = WV.load_subset(name)
    except FileNotFoundError as e:
        print(f"skip {name}: {e}")

for name, (tok, emb) in EMB.items():
    assert emb.vocab.index("<pad>") == PAD_IDX and emb.vocab.index("<unk>") == UNK_IDX
    print(f"{name:11s} {emb.vectors.shape[0]:,} words x {emb.vectors.shape[1]}d")

EMB_NAME = "ft-crawl"  # the one to use; change here to switch
tok, emb = EMB[EMB_NAME]


def emb_matrix(name, standardise=True):
    """(V, 300) float32 tensor for embedding `name`; its rows match the ids in IDS[name].
    standardise: z-score each dimension over the vocabulary rows, as the baseline's StandardScaler does;
    the <pad> row stays 0. Raw vectors are tiny (|component| ~0.05) and the models train poorly on them.
    """
    v = EMB[name][1].vectors
    if standardise:
        words = v[2:]  # all rows except <pad> and <unk>
        v = (v - words.mean(0)) / words.std(0)  # new array: emb.vectors is not modified
        v[PAD_IDX] = 0.0
    return torch.tensor(v, dtype=torch.float32)


E = emb_matrix(EMB_NAME)
E.shape

## 4. Tokenise once

Every text becomes a list of token ids, once per embedding (each embedding has its own tokeniser and vocabulary). The ids are cached in `IDS[name]` for the train split and `IDS_TEST[name]` for the held-out test split. **`IDS_TEST` is only used in section 10 (final test).**

In [ ]:
IDS = {name: [t.ids(text) for text in texts] for name, (t, _) in EMB.items()}
IDS_TEST = {name: [t.ids(text) for text in D.test["text"]] for name, (t, _) in EMB.items()}
y_test = D.y_test

ex = texts[0]
print(ex[:300], "...\n")
print("tokens:", tok.tokens(ex)[:25])
print("ids:   ", IDS[EMB_NAME][0][:25])

lengths = pd.Series([len(s) for s in IDS[EMB_NAME]])
unk_share = np.mean([i == UNK_IDX for s in IDS[EMB_NAME] for i in s])
print(
    f"\ntokens per text: median {lengths.median():.0f}, 95th pct {lengths.quantile(.95):.0f}, max {lengths.max()}"
)
print(f"<unk> share of tokens: {unk_share:.4%}")
assert lengths.min() > 0

## 5. Dataset and batches

`collate` pads each batch to its longest text with `PAD_IDX` and also returns the true lengths, which `pack_padded_sequence` needs, so the LSTM never reads padding.

Each batch is `x` (B, T) long, `lengths` (B,) long and `y` (B,) long. `make_loader(idx, emb_name)` indexes the train split; `test=True` indexes the test split instead. `by_prompt=True` makes each batch one whole prompt (18 texts, one per model), which prompt-centring needs (section 13).

In [ ]:
class TextDataset(Dataset):
    def __init__(self, seqs, labels):
        self.seqs = [torch.tensor(s, dtype=torch.long) for s in seqs]
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return self.seqs[i], self.labels[i]


def collate(batch):
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs], dtype=torch.long)
    x = pad_sequence(seqs, batch_first=True, padding_value=PAD_IDX)
    return x, lengths, torch.stack(labels)


class PromptBatches:
    """Batch sampler: one batch per prompt (all its texts), prompts in random order each epoch if shuffle."""

    def __init__(self, prompt_ids, shuffle):
        self.batches = [np.flatnonzero(prompt_ids == p).tolist() for p in pd.unique(prompt_ids)]
        self.shuffle = shuffle

    def __len__(self):
        return len(self.batches)

    def __iter__(self):
        order = (
            torch.randperm(len(self.batches)).tolist() if self.shuffle else range(len(self.batches))
        )
        return (self.batches[i] for i in order)


def make_loader(idx, emb_name=EMB_NAME, batch_size=32, shuffle=False, test=False, by_prompt=False):
    """DataLoader over the texts at positions `idx` (e.g. one fold), tokenised for `emb_name`.
    Positions refer to the train split, or to the test split if `test=True`.
    by_prompt: each batch is all the texts of one prompt (batch_size is ignored), for prompt-centring.
    """
    ids, labels = (IDS_TEST, y_test) if test else (IDS, y)
    ds = TextDataset([ids[emb_name][i] for i in idx], labels[idx])
    if by_prompt:
        prompt_ids = (D.test["prompt_id"].to_numpy() if test else groups)[idx]
        return DataLoader(ds, batch_sampler=PromptBatches(prompt_ids, shuffle), collate_fn=collate)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, collate_fn=collate)


x, lens, yb = next(iter(make_loader(np.arange(64), shuffle=True)))
print("x", tuple(x.shape), x.dtype, "| lengths", tuple(lens.shape), "| y", tuple(yb.shape))

## 6. Folds

These are the team's shared folds (`shared/folds.json`): 5 folds grouped by prompt, with 10 repeats available. Neural models are slow, so we use 1 repeat (5 folds) by default.

Inside each outer training fold, `H.inner_split` holds out 20% of the prompts as an **inner validation set** for early stopping and model selection. The outer test fold is only used for the reported score.

In [ ]:
FOLDS = H.load_folds(
    D.train["id"].tolist(), repeats=1
)  # list of (repeat, fold, train_idx, test_idx)

r, k, tr, te = FOLDS[0]
itr, iva = H.inner_split(tr, groups, seed=H.SEED + r)
print(f"{len(FOLDS)} folds; fold 0: inner train {len(itr)}, inner val {len(iva)}, test {len(te)}")
P = lambda idx: set(groups[idx])
assert (
    not (P(itr) & P(iva)) and not (P(itr) & P(te)) and not (P(iva) & P(te))
), "prompts leak between splits"

## 7. The model

Frozen pretrained embeddings → LSTM (`num_layers` stacked layers of `hidden_size` units) → **mean over the top layer's outputs at every real token** → dropout → one linear layer giving 18 logits (no softmax: the loss applies it).

- **Mean pooling:** the text is represented by the average of the LSTM's output at every token, not only its final state. Information from the whole text counts equally, as in the mean-pooled embedding baseline. `pack_padded_sequence` / `pad_packed_sequence` keep padding out: padded positions come back as zeros and the sum is divided by each text's real length.
- **Dropout between layers** (`dropout`, a required argument searched in the grid, active only in training): between stacked LSTM layers (`nn.LSTM(dropout=...)`, which only applies when `num_layers > 1`) and between the LSTM and the linear layer, so 1-layer models are regularised too.
- **Frozen by default:** `freeze=False` makes the embeddings trainable (section 12). The pretrained vectors are then only the starting point. `from_pretrained` uses `E` itself as the weight, so pass `E.clone()` when training them; otherwise every fold would start from the previous fold's updated vectors.
- **Prompt-centring** (`centre=True`, section 13): the pooled vectors minus their batch mean. The training loop then gives the model one prompt per batch, so this subtracts the prompt's mean, as `prompt_centre` does for the probe.

The contract the training loop needs: `model(x, lengths)` returns logits of shape (B, N_CLASSES), and a no-argument `make_model()` builds a fresh model (one per fold).

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(
        self,
        E,
        hidden_size,
        num_layers,
        dropout,
        n_classes=N_CLASSES,
        pad_idx=PAD_IDX,
        freeze=True,
        centre=False,
    ):
        super().__init__()
        self.centre = (
            centre  # subtract the batch mean of the pooled vectors; batches must be one prompt each
        )
        self.emb = nn.Embedding.from_pretrained(E, freeze=freeze, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            E.shape[1],
            hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )  # between stacked LSTM layers
        self.drop = nn.Dropout(dropout)  # between the LSTM and the linear layer
        self.out = nn.Linear(hidden_size, n_classes)

    def forward(self, x, lengths):
        packed = pack_padded_sequence(
            self.emb(x), lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        outputs, _ = self.lstm(packed)
        outputs, _ = pad_packed_sequence(
            outputs, batch_first=True
        )  # (B, T, hidden_size), zeros after each text ends
        pooled = outputs.sum(1) / lengths.to(outputs.device).unsqueeze(
            1
        )  # mean over real tokens -> (B, hidden_size)
        if self.centre:  # batch = one prompt, so this is prompt-centring
            pooled = pooled - pooled.mean(0, keepdim=True)
        return self.out(self.drop(pooled))  # (B, N_CLASSES)

## 8. Training and evaluation

`train_fold` trains one model with **Adam** and an **exponentially decaying learning rate** (`lr` × `lr_decay`^epoch):
1. train on the inner train split;
2. after each epoch, compute the loss on the inner validation split, keep the best weights, and stop after `patience` epochs without improvement;
3. return the model with its best weights.

`run_cv` does this for each outer fold and scores the model on the outer test fold (balanced accuracy, accuracy, macro F1). `verbose` sets how much progress is printed: `0` only the final line, `1` (default) one line per fold, `2` also one line per epoch (learning rate, train and validation loss, `*` when the validation loss improved). Each fold row also stores the train and validation loss and the learning rate per epoch (`history`), used in section 9b to look at overfitting.

`run_cv(..., seed=s)` retrains with another initialisation and batch order on the same folds (section 14), and `shuffle_test=True` also scores each test fold with the tokens of every text shuffled (section 15).

`run_cv` returns a dict in the team's result format, so `H.save_result(res)` writes it to `shared/results/` and `H.compare(res, baseline)` gives the Bayesian comparison (ROPE 1 pp).

In [ ]:
def predict_logprobs(model, loader, device=DEVICE):
    """Log-probabilities in dataset order (prompt batches come out grouped, so rows are put back)."""
    model.eval()
    order = np.concatenate(
        [np.asarray(b) for b in loader.batch_sampler]
    )  # dataset position of each output row
    out = []
    with torch.no_grad():
        for x, lens, _ in loader:
            out.append(torch.log_softmax(model(x.to(device), lens), -1).cpu())
    logp = torch.cat(out).numpy()
    res = np.empty_like(logp)
    res[order] = logp
    return res


def train_fold(
    make_model,
    tr,
    va,
    emb_name=EMB_NAME,
    lr=1e-3,
    lr_decay=0.9,
    weight_decay=0.0,
    epochs=30,
    patience=5,
    batch_size=32,
    device=DEVICE,
    verbose=False,
):
    """Train on train-split positions `tr`, early-stop on the loss of `va`. Returns (model, best_epoch, history).
    verbose=True prints one line per epoch."""
    model = make_model().to(device)
    opt = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=weight_decay
    )
    sched = torch.optim.lr_scheduler.ExponentialLR(
        opt, gamma=lr_decay
    )  # lr is multiplied by lr_decay each epoch
    loss_fn = nn.CrossEntropyLoss()
    by_prompt = getattr(model, "centre", False)  # a prompt-centred model needs one prompt per batch
    train_dl = make_loader(tr, emb_name, batch_size, shuffle=True, by_prompt=by_prompt)
    val_dl = make_loader(va, emb_name, 256, by_prompt=by_prompt)

    best, best_state, best_epoch, history = float("inf"), None, -1, []
    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        total = 0.0
        for x, lens, yb in train_dl:
            opt.zero_grad()
            loss = loss_fn(model(x.to(device), lens), yb.to(device))
            loss.backward()
            opt.step()
            total += loss.item() * len(yb)
        val_logp = predict_logprobs(model, val_dl, device)
        val_loss = float(-val_logp[np.arange(len(va)), y[va]].mean())
        history.append(
            {
                "epoch": epoch,
                "lr": sched.get_last_lr()[0],
                "train_loss": total / len(tr),
                "val_loss": val_loss,
            }
        )
        sched.step()
        improved = val_loss < best
        if improved:
            best, best_state, best_epoch = val_loss, copy.deepcopy(model.state_dict()), epoch
        if verbose:
            h = history[-1]
            print(
                f"    epoch {epoch:2d}  lr {h['lr']:.2e}  train {h['train_loss']:.4f}  val {val_loss:.4f}"
                f"{'  *' if improved else ''}  ({time.time() - t0:.0f}s)",
                flush=True,
            )
        if not improved and epoch - best_epoch >= patience:
            if verbose:
                print(
                    f"    early stop: no val improvement for {patience} epochs, keeping epoch {best_epoch}"
                )
            break

    model.load_state_dict(best_state)
    return model, best_epoch, history


def shuffled_loader(idx, emb_name, by_prompt=False, seed=0):
    """make_loader(idx, emb_name, 256, by_prompt=...) with each text's token ids in random order (same tokens,
    no word order): mean pooling can't tell the difference, so any drop shows how much a model uses order.
    """
    dl = make_loader(idx, emb_name, 256, by_prompt=by_prompt)
    rng = np.random.default_rng(seed)
    dl.dataset.seqs = [s[torch.from_numpy(rng.permutation(len(s)))] for s in dl.dataset.seqs]
    return dl


def run_cv(
    make_model,
    name,
    emb_name=EMB_NAME,
    folds=None,
    verbose=1,
    seed=0,
    shuffle_test=False,
    **train_kw,
):
    """CV on the shared folds (early stopping on an inner split of each training fold).
    `name` follows the team convention, e.g. "ft-crawl-lstm-h100-l2/linear".
    verbose: 0 = summary line only, 1 = one line per fold, 2 = also one line per epoch.
    seed: 0 reproduces every saved result; 1, 2, ... give other initialisations and batch orders on the same
    folds and inner splits (section 14).
    shuffle_test: also score each test fold with its texts' tokens shuffled; stored in the fold rows as
    balanced_accuracy_shuffled etc. (section 15)."""
    folds = FOLDS if folds is None else folds
    rows, oof_pred, oof_proba = [], np.full(len(y), -1), np.zeros((len(y), N_CLASSES))
    t0 = time.time()
    for r, k, tr, te in folds:
        H.set_seed(H.SEED + r + 1000 * seed)
        itr, iva = H.inner_split(tr, groups, seed=H.SEED + r)
        if verbose >= 2:
            print(f"  fold {k}: train {len(itr)}, val {len(iva)}, test {len(te)}", flush=True)
        model, best_epoch, history = train_fold(
            make_model, itr, iva, emb_name, verbose=verbose >= 2, **train_kw
        )
        logp = predict_logprobs(
            model, make_loader(te, emb_name, 256, by_prompt=getattr(model, "centre", False))
        )
        pred = logp.argmax(1)
        shuffled = {}
        if shuffle_test:  # the same model on the same test texts, tokens in random order
            sl = shuffled_loader(te, emb_name, getattr(model, "centre", False), seed=H.SEED + r + k)
            shuffled = {
                f"{m}_shuffled": v
                for m, v in H.scores(y[te], predict_logprobs(model, sl).argmax(1)).items()
            }
        rows.append(
            {
                "repeat": r,
                "fold": k,
                **H.scores(y[te], pred),
                **shuffled,
                "best_epoch": best_epoch,
                "history": history,
                "n_test": len(te),
                "n_total": len(tr) + len(te),
            }
        )
        if r == folds[0][0]:
            oof_pred[te], oof_proba[te] = pred, np.exp(logp)
        if verbose:
            print(
                f"  fold {k}: bal.acc {rows[-1]['balanced_accuracy']:.4f}  (best epoch {best_epoch}, "
                f"{time.time() - t0:.0f}s)",
                flush=True,
            )
    res = {
        "name": name,
        "model": "torch",
        "folds": rows,
        "oof_pred": oof_pred,
        "oof_proba": oof_proba,
        "seconds": time.time() - t0,
        "runs": len({f[0] for f in folds}),
        "config": {"emb": emb_name, **train_kw, **({"seed": seed} if seed else {})},
    }
    ba = H.fold_scores(res)
    print(f"{name}: bal.acc {ba.mean():.4f} ± {ba.std():.4f} over {len(rows)} folds")
    return res

### Smoke test

Checks the output shape, then runs a small LSTM for 2 epochs on one fold. It only shows the pipeline runs end to end; it is **not** an experiment, and nothing is saved.

In [ ]:
_m = LSTMClassifier(E, hidden_size=50, num_layers=2, dropout=0.3)
_x, _lens, _ = next(iter(make_loader(np.arange(8))))
assert _m(_x, _lens).shape == (8, N_CLASSES)

_smoke = run_cv(
    lambda: LSTMClassifier(E, 50, 1, dropout=0.3), "smoke", folds=FOLDS[:1], epochs=2, verbose=2
)

## 9. Hyperparameter search

`grid_search(emb_name)` runs the CV above (5 folds × 1 repeat over the whole train split) for every combination of
- hidden units: 64, 128, 256;
- LSTM layers: 1, 2 (3+ layers did worse in earlier checks);
- dropout: 0.2, 0.5;

that is 12 configs, with Adam and the decaying learning rate (`lr=1e-3`, `lr_decay=0.9`) fixed. Override any of them, e.g. `grid_search(..., dropout=(0.3,), lr=...)`.

It returns a summary table sorted by mean balanced accuracy and a dict `{(hidden, layers, dropout): result}` with the full results. The table is also written to `part2/lstm_grid_<emb>.csv`, and every config's CV result to `shared/results/`, so they survive a kernel restart (section 14 compares the configs).

Each fold's score comes from a fold the model never saw, but picking the best of 12 configs on those same scores makes the winner's CV number slightly optimistic. The test split (section 10) gives the unbiased number.

In [ ]:
HIDDEN = (64, 128, 256)
LAYERS = (1, 2)
DROPOUT = (0.2, 0.5)


def grid_search(
    emb_name,
    hidden=HIDDEN,
    layers=LAYERS,
    dropout=DROPOUT,
    folds=None,
    save_csv=True,
    save_results=True,
    verbose=1,
    **train_kw,
):
    """verbose: 0 = one line per config, 1 = also one line per fold, 2 = also one line per epoch.
    save_results: save every config's CV result to shared/results/ (section 14 compares them)."""
    E_ = emb_matrix(emb_name)
    configs = [(h, l, d) for h in hidden for l in layers for d in dropout]
    rows, results, t0 = [], {}, time.time()
    for i, (h, l, d) in enumerate(configs, 1):
        print(
            f"[{i}/{len(configs)}] {emb_name}: hidden {h}, layers {l}, dropout {d} ({(time.time() - t0) / 60:.1f} min elapsed)",
            flush=True,
        )
        name = f"{emb_name}-lstm-h{h}-l{l}-d{d}/linear"
        res = run_cv(
            lambda h=h, l=l, d=d: LSTMClassifier(E_, h, l, dropout=d),
            name,
            emb_name,
            folds=folds,
            verbose=verbose,
            **train_kw,
        )
        res["config"].update(hidden=h, layers=l, dropout=d, pooling="mean")
        if save_results:
            H.save_result(res)
        results[(h, l, d)] = res
        f = pd.DataFrame(res["folds"])
        rows.append(
            {
                "hidden": h,
                "layers": l,
                "dropout": d,
                "bal_acc": f["balanced_accuracy"].mean(),
                "bal_acc_std": f["balanced_accuracy"].std(ddof=0),
                "accuracy": f["accuracy"].mean(),
                "macro_f1": f["macro_f1"].mean(),
                "best_epoch": f["best_epoch"].mean(),
                "minutes": res["seconds"] / 60,
            }
        )
    summary = pd.DataFrame(rows).sort_values("bal_acc", ascending=False).reset_index(drop=True)
    if save_csv:
        summary.to_csv(ROOT / "part2" / f"lstm_grid_{emb_name}.csv", index=False)
    return summary, results

### 9b. Overfitting

`plot_curves(res)` draws, for each fold, the training loss and the inner-validation loss per epoch; the dashed line is the epoch whose weights were kept. Overfitting shows up as the training loss still falling while the validation loss turns back up. The table below the plot gives both losses at the kept epoch and their gap.

In [ ]:
TRAIN_C, VAL_C, MUTED = "#2a78d6", "#eb6834", "#8a8a85"  # categorical slots 1-2; neutral for guides


def plot_curves(res):
    folds = res["folds"]
    fig, axes = plt.subplots(
        1, len(folds), figsize=(3.2 * len(folds), 3), sharey=True, squeeze=False
    )
    for ax, f in zip(axes[0], folds):
        h = pd.DataFrame(f["history"])
        ax.plot(h["epoch"], h["train_loss"], color=TRAIN_C, lw=2, label="train")
        ax.plot(h["epoch"], h["val_loss"], color=VAL_C, lw=2, label="inner validation")
        ax.axvline(f["best_epoch"], color=MUTED, lw=1, ls="--")
        ax.set_title(f"fold {f['fold']} · bal.acc {f['balanced_accuracy']:.3f}", fontsize=10)
        ax.set_xlabel("epoch")
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        ax.grid(axis="y", color="#e6e6e3", lw=0.8)
        ax.spines[["top", "right"]].set_visible(False)
    axes[0][0].set_ylabel("cross-entropy loss")
    axes[0][0].legend(frameon=False, fontsize=9)
    fig.suptitle(res["name"], fontsize=11)
    fig.tight_layout()
    plt.show()

    at_best = pd.DataFrame(
        [
            {
                "fold": f["fold"],
                "best_epoch": f["best_epoch"],
                "train_loss": f["history"][f["best_epoch"]]["train_loss"],
                "val_loss": f["history"][f["best_epoch"]]["val_loss"],
                "outer_bal_acc": f["balanced_accuracy"],
            }
            for f in folds
        ]
    )
    at_best["gap"] = at_best["val_loss"] - at_best["train_loss"]
    return at_best.round(3)


plot_curves(_smoke)  # smoke-test run, just to show the plot; replace with a real result

In [ ]:
def plot_grid_curves(results):
    """Train and inner-validation loss per epoch for every config of a grid search: one panel per
    config (rows = hidden size, columns = layers and dropout), one thin line per fold.
    `results` is the dict returned by grid_search. Saves to part2/graphs/lstm/grid_curves_<emb>.png.
    """
    keys = sorted(results)
    hiddens = sorted({h for h, _, _ in keys})
    cols = sorted({(l, d) for _, l, d in keys})
    fig, axes = plt.subplots(
        len(hiddens),
        len(cols),
        figsize=(3.2 * len(cols), 2.6 * len(hiddens)),
        sharex=True,
        sharey=True,
        squeeze=False,
    )
    for h, l, d in keys:
        res = results[(h, l, d)]
        ax = axes[hiddens.index(h), cols.index((l, d))]
        for i, f in enumerate(res["folds"]):
            hist = pd.DataFrame(f["history"])
            ax.plot(
                hist["epoch"],
                hist["train_loss"],
                color=TRAIN_C,
                lw=1,
                alpha=0.7,
                label="train" if i == 0 else None,
            )
            ax.plot(
                hist["epoch"],
                hist["val_loss"],
                color=VAL_C,
                lw=1,
                alpha=0.7,
                label="inner validation" if i == 0 else None,
            )
        ax.set_title(f"h{h} · l{l} · d{d} · bal.acc {H.fold_scores(res).mean():.3f}", fontsize=10)
        ax.grid(axis="y", color="#e6e6e3", lw=0.8)
        ax.spines[["top", "right"]].set_visible(False)
    for ax in axes[-1]:
        ax.set_xlabel("epoch")
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    for ax in axes[:, 0]:
        ax.set_ylabel("cross-entropy loss")
    axes[0, 0].legend(frameon=False, fontsize=9)

    emb_name = results[keys[0]]["config"]["emb"]
    fig.suptitle(f"{emb_name}: LSTM grid, train vs inner-validation loss per fold", fontsize=11)
    fig.tight_layout()
    path = ROOT / "part2" / "graphs" / "lstm" / f"grid_curves_{emb_name}.png"
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    return path

## 10. Final test (run once)

`final_test` is the single evaluation on the held-out test split, following the team's `H.final_test` convention: hold out 20% of the train split's prompts for early stopping, train on the rest with the chosen config, and score the test split once. **Only call it after the config is chosen by CV.**

In [ ]:
def final_test(emb_name, hidden, layers, dropout, verbose=True, **train_kw):
    H.set_seed(H.SEED)
    itr, iva = H.inner_split(np.arange(len(y)), groups, seed=H.SEED)
    E_ = emb_matrix(emb_name)
    model, best_epoch, history = train_fold(
        lambda: LSTMClassifier(E_, hidden, layers, dropout),
        itr,
        iva,
        emb_name,
        verbose=verbose,
        **train_kw,
    )
    pred = predict_logprobs(
        model, make_loader(np.arange(len(y_test)), emb_name, 256, test=True)
    ).argmax(1)
    return {
        "name": f"{emb_name}-lstm-h{hidden}-l{layers}-d{dropout}/linear",
        "emb": emb_name,
        "hidden": hidden,
        "layers": layers,
        "dropout": dropout,
        **H.scores(y_test, pred),
        "best_epoch": best_epoch,
        "history": history,
        "pred": pred.tolist(),
    }

## 11. Full pipeline per embedding

`run_pipeline(emb_name)`:
1. runs the 12-config grid (section 9) and shows the table;
2. picks the config with the highest mean balanced accuracy and saves its CV result to `shared/results/<emb>-lstm-h<h>-l<l>-d<d>__linear.json` (the team format, so it can be compared with the baselines);
3. plots its loss curves (section 9b);
4. runs the final test (section 10) and writes the scores to `part2/lstm_test_<emb>.json`.

`hidden`, `layers` and `dropout` (tuples of values to search) go to the grid; `**train_kw` (e.g. `weight_decay=1e-4`) are passed on to the grid and the final test. `verbose` works as in `grid_search` (`2` prints every epoch). Compare with a baseline afterwards:
```python
R = H.load_results()
H.compare(R["emb-ft-crawl-mean/logreg"], ft_results[(h, l, d)])   # P(A>B), P(ROPE), P(B>A)
```

In [ ]:
import json


def run_pipeline(
    emb_name, hidden=HIDDEN, layers=LAYERS, dropout=DROPOUT, folds=None, verbose=1, **train_kw
):
    """Grid arguments (hidden, layers, dropout, folds) go to the grid only; **train_kw (lr, epochs, ...) to both."""
    summary, results = grid_search(
        emb_name, hidden, layers, dropout, folds=folds, verbose=verbose, **train_kw
    )
    display(summary.round(4))

    h, l, d = (
        int(summary.loc[0, "hidden"]),
        int(summary.loc[0, "layers"]),
        float(summary.loc[0, "dropout"]),
    )
    best = results[(h, l, d)]
    print(f"chosen: hidden {h}, layers {l}, dropout {d} -> saved {H.save_result(best)}")
    display(plot_curves(best))

    print("final test: training on the whole train split")
    test = final_test(emb_name, h, l, d, verbose=verbose >= 1, **train_kw)
    (ROOT / "part2" / f"lstm_test_{emb_name}.json").write_text(json.dumps(test))
    print(
        f"TEST {test['name']}: bal.acc {test['balanced_accuracy']:.4f}, acc {test['accuracy']:.4f}, "
        f"macro F1 {test['macro_f1']:.4f}"
    )
    return summary, results, test

**Each of the two cells below takes roughly 1–2 hours on CPU (9 configs × 5 folds × up to 30 epochs) and evaluates on the test split.** Run them when the model in section 7 is final.

In [ ]:
ft_summary, ft_results, ft_test = run_pipeline("ft-crawl")

In [ ]:
plot_grid_curves(ft_results)

In [ ]:
glove_summary, glove_results, glove_test = run_pipeline("glove-840b")

In [ ]:
plot_grid_curves(glove_results)

## 12. Frozen vs trainable embeddings

`frozen_vs_trainable(emb_name)` takes the grid winner stored by the last `run_pipeline(emb_name)`:
- its (hidden, layers, dropout) come from row 0 of `part2/lstm_grid_<emb>.csv`;
- its frozen CV result, including the training settings (`lr`, `epochs`, …) in its `config`, comes from `shared/results/`.

It retrains that config with `freeze=False` on the same folds and seeds and compares the two with `H.compare` (A = frozen, B = trainable, ROPE 1 pp). Nothing is hard-coded, so re-running the grid is picked up automatically. The trainable result is saved as `<emb>-lstm-h<h>-l<l>-d<d>-trainemb/linear`.

Unfreezing adds about 19.5k × 300 ≈ 5.9M parameters for about 1.7k training texts per fold, so look at `plot_curves(TRAIN_EMB[emb])` for overfitting. Only words that occur in the training texts get updated; the rest keep their pretrained vectors.

In [ ]:
def frozen_vs_trainable(emb_name, folds=None, save=True, verbose=1):
    """Retrain the stored grid winner for `emb_name` with trainable embeddings and compare it with the
    frozen run on the same folds. Returns (summary row, trainable result), or (None, None) if the
    grid hasn't been run for this embedding."""
    csv = ROOT / "part2" / f"lstm_grid_{emb_name}.csv"
    if not csv.exists():
        print(f"skip {emb_name}: {csv.name} not found, run run_pipeline('{emb_name}') first")
        return None, None
    best = pd.read_csv(csv).iloc[0]
    frozen_name = f"{emb_name}-lstm-h{int(best['hidden'])}-l{int(best['layers'])}-d{float(best['dropout'])}/linear"
    frozen = H.load_results().get(frozen_name)
    if frozen is None:
        print(
            f"skip {emb_name}: {frozen_name} not in shared/results, run run_pipeline('{emb_name}') first"
        )
        return None, None

    train_kw = dict(
        frozen["config"]
    )  # everything except the model settings = the training settings
    h, l, d = train_kw.pop("hidden"), train_kw.pop("layers"), train_kw.pop("dropout")
    for k in ("emb", "pooling", "freeze"):
        train_kw.pop(k, None)

    folds = FOLDS if folds is None else folds
    E_ = emb_matrix(emb_name)
    res = run_cv(
        lambda: LSTMClassifier(
            E_.clone(), h, l, d, freeze=False
        ),  # clone: each fold starts from the pretrained vectors
        frozen_name.replace("/linear", "-trainemb/linear"),
        emb_name,
        folds=folds,
        verbose=verbose,
        **train_kw,
    )
    res["config"].update(hidden=h, layers=l, dropout=d, pooling="mean", freeze=False)
    if save:
        H.save_result(res)

    keep = {(r, k) for r, k, _, _ in folds}  # pair on the folds actually run
    frozen = {**frozen, "folds": [f for f in frozen["folds"] if (f["repeat"], f["fold"]) in keep]}
    c = H.compare(frozen, res)
    row = {
        "emb": emb_name,
        "config": f"h{h} l{l} d{d}",
        "frozen": H.fold_scores(frozen).mean(),
        "trainable": H.fold_scores(res).mean(),
        "diff_pp": 100 * c["mean_diff"],
        "P(frozen better)": c["p_a_better"],
        "P(ROPE)": c["p_rope"],
        "P(trainable better)": c["p_b_better"],
    }
    return row, res

In [ ]:
TRAIN_EMB, rows = {}, []
for emb_name in ["ft-crawl", "glove-840b"]:
    row, res = frozen_vs_trainable(emb_name)
    if res is not None:
        TRAIN_EMB[emb_name], rows = res, rows + [row]
pd.DataFrame(rows).round(4)

## 13. Vanilla vs prompt-centred

For the probe, subtracting each prompt's mean vector gave +12 pp, because an averaged text vector is mostly topic. Here the same is done inside the LSTM: with `centre=True` the model subtracts the batch mean from the pooled vectors, and the training loop makes each batch one whole prompt (18 texts, one per model). The folds and the inner split are grouped by prompt, so a prompt is never split. Only prompt ids are used, never labels: at test time, a prompt's other texts are used unlabelled, as in the probe.

`vanilla_vs_centred(emb_name)` takes the **best version from section 12**, the frozen or trainable grid winner, whichever has the higher CV balanced accuracy. It retrains it with `centre=True` on the same folds and seeds and compares the two with `H.compare` (A = vanilla, B = centred). Like section 12, it reads the stored results, so nothing is hard-coded. The figure (`part2/graphs/lstm/vanilla_vs_centred_<emb>.png`) shows the loss curves of both and the per-fold balanced accuracy, paired by fold.

Caveat: centred batches are 18 texts from one prompt, while vanilla batches are 32 random texts, so batch composition differs too.

In [ ]:
INK = "#2b2b28"  # text-like dark grey for the mean line


def plot_vanilla_vs_centred(vanilla, centred, emb_name, cmp):
    """Loss curves of both runs (one thin line per fold) and their per-fold balanced accuracy, paired by fold.
    `cmp` is H.compare(vanilla, centred). Saves to part2/graphs/lstm/vanilla_vs_centred_<emb>.png.
    """
    fig = plt.figure(figsize=(13, 3.6))
    ax_v = fig.add_subplot(1, 3, 1)
    ax_c = fig.add_subplot(1, 3, 2, sharey=ax_v)
    ax_s = fig.add_subplot(1, 3, 3)
    for ax, res, title in [(ax_v, vanilla, "vanilla"), (ax_c, centred, "prompt-centred")]:
        for i, f in enumerate(res["folds"]):
            hist = pd.DataFrame(f["history"])
            ax.plot(
                hist["epoch"],
                hist["train_loss"],
                color=TRAIN_C,
                lw=1,
                alpha=0.7,
                label="train" if i == 0 else None,
            )
            ax.plot(
                hist["epoch"],
                hist["val_loss"],
                color=VAL_C,
                lw=1,
                alpha=0.7,
                label="inner validation" if i == 0 else None,
            )
        ax.set_title(f"{title} · bal.acc {H.fold_scores(res).mean():.3f}", fontsize=10)
        ax.set_xlabel("epoch")
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax_v.set_ylabel("cross-entropy loss")
    ax_v.legend(frameon=False, fontsize=9)

    a, b = H.fold_scores(vanilla), H.fold_scores(centred)
    for i, (va_, ce_) in enumerate(zip(a, b)):
        ax_s.plot(
            [0, 1],
            [va_, ce_],
            color=MUTED,
            lw=1,
            marker="o",
            ms=4,
            label="fold" if i == 0 else None,
        )
    ax_s.plot([0, 1], [a.mean(), b.mean()], color=INK, lw=2, marker="o", ms=8, label="mean")
    ax_s.set_xticks([0, 1], ["vanilla", "prompt-centred"])
    ax_s.set_xlim(-0.3, 1.3)
    ax_s.set_ylabel("balanced accuracy")
    ax_s.set_title(
        f"per fold · Δ {100 * cmp['mean_diff']:+.1f} pp · P(centred better) {cmp['p_b_better']:.2f}",
        fontsize=10,
    )
    ax_s.legend(frameon=False, fontsize=9)

    for ax in (ax_v, ax_c, ax_s):
        ax.grid(axis="y", color="#e6e6e3", lw=0.8)
        ax.spines[["top", "right"]].set_visible(False)
    fig.suptitle(f"{emb_name}: {vanilla['name']} vs prompt-centred", fontsize=11)
    fig.tight_layout()
    path = ROOT / "part2" / "graphs" / "lstm" / f"vanilla_vs_centred_{emb_name}.png"
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    return path


def vanilla_vs_centred(emb_name, folds=None, save=True, verbose=1):
    """Retrain the best version from section 12 (frozen or trainable grid winner) with prompt-centring and
    compare it with the vanilla run on the same folds. Returns (summary row, centred result), or (None, None).
    """
    csv = ROOT / "part2" / f"lstm_grid_{emb_name}.csv"
    if not csv.exists():
        print(f"skip {emb_name}: {csv.name} not found, run run_pipeline('{emb_name}') first")
        return None, None
    best = pd.read_csv(csv).iloc[0]
    frozen_name = f"{emb_name}-lstm-h{int(best['hidden'])}-l{int(best['layers'])}-d{float(best['dropout'])}/linear"
    R = H.load_results()
    candidates = {
        v: R[n]
        for v, n in [
            ("frozen", frozen_name),
            ("trainable", frozen_name.replace("/linear", "-trainemb/linear")),
        ]
        if n in R
    }
    if not candidates:
        print(
            f"skip {emb_name}: {frozen_name} not in shared/results, run run_pipeline('{emb_name}') first"
        )
        return None, None
    if "trainable" not in candidates:
        print(f"{emb_name}: no trainable-embedding result (section 12), using the frozen one")
    version = max(candidates, key=lambda v: H.fold_scores(candidates[v]).mean())
    vanilla = candidates[version]

    train_kw = dict(
        vanilla["config"]
    )  # everything except the model settings = the training settings
    h, l, d = train_kw.pop("hidden"), train_kw.pop("layers"), train_kw.pop("dropout")
    freeze = train_kw.pop("freeze", True)
    for k in ("emb", "pooling", "centre"):
        train_kw.pop(k, None)

    folds = FOLDS if folds is None else folds
    E_ = emb_matrix(emb_name)
    res = run_cv(
        lambda: LSTMClassifier(E_.clone(), h, l, d, freeze=freeze, centre=True),
        vanilla["name"].replace("/linear", "-centred/linear"),
        emb_name,
        folds=folds,
        verbose=verbose,
        **train_kw,
    )
    res["config"].update(hidden=h, layers=l, dropout=d, pooling="mean", freeze=freeze, centre=True)
    if save:
        H.save_result(res)

    keep = {(r, k) for r, k, _, _ in folds}  # pair on the folds actually run
    vanilla = {
        **vanilla,
        "folds": [f for f in vanilla["folds"] if (f["repeat"], f["fold"]) in keep],
    }
    c = H.compare(vanilla, res)
    plot_vanilla_vs_centred(vanilla, res, emb_name, c)
    row = {
        "emb": emb_name,
        "version": version,
        "config": f"h{h} l{l} d{d}",
        "vanilla": H.fold_scores(vanilla).mean(),
        "centred": H.fold_scores(res).mean(),
        "diff_pp": 100 * c["mean_diff"],
        "P(vanilla better)": c["p_a_better"],
        "P(ROPE)": c["p_rope"],
        "P(centred better)": c["p_b_better"],
    }
    return row, res

In [ ]:
CENTRED, rows = {}, []
for emb_name in ["ft-crawl", "glove-840b"]:
    row, res = vanilla_vs_centred(emb_name)
    if res is not None:
        CENTRED[emb_name], rows = res, rows + [row]
pd.DataFrame(rows).round(4)

## 14. Uncertainty (answer 5)

How solid are the choices above? Four checks, each on the shared folds and each read from the stored results, so a re-run of sections 11–13 is picked up automatically:
1. **Grid report** (no training): is the grid winner actually better than the other configs, and which hyperparameters matter?
2. **Selection optimism** (no training): the winner's CV score against its test score.
3. **Seed variance** (about 10–20 min per system and seed): the final systems retrained with other seeds on the same folds. How much does a result move with the initialisation alone, and do the conclusions of sections 12–13 hold?
4. **More repeats** (about 2 CV runs per embedding): the best LSTM on 3 repeats (15 folds), compared with the best embed-and-pool system on the same folds.

The helpers below find the systems: the **grid winner** (row 0 of the grid CSV), the **best vanilla** version (frozen or trainable, section 12) and its **centred** version (section 13). `rebuild(res)` retrains any saved system exactly as it was trained.

In [ ]:
import json

from shared import analysis as A

GRAPHS = ROOT / "part2" / "graphs" / "lstm"
MODEL_KEYS = ("emb", "hidden", "layers", "dropout", "pooling", "freeze", "centre", "seed")


def save_fig(fig, filename):
    GRAPHS.mkdir(parents=True, exist_ok=True)
    fig.savefig(GRAPHS / filename, dpi=150, bbox_inches="tight")
    return GRAPHS / filename


def grid_winner_name(emb_name):
    """Name of the frozen grid winner stored by the last run_pipeline (row 0 of the grid CSV), or None."""
    csv = ROOT / "part2" / f"lstm_grid_{emb_name}.csv"
    if not csv.exists():
        return None
    b = pd.read_csv(csv).iloc[0]
    return f"{emb_name}-lstm-h{int(b['hidden'])}-l{int(b['layers'])}-d{float(b['dropout'])}/linear"


def best_vanilla_name(emb_name, R):
    """The frozen or trainable grid winner, whichever has the higher CV mean (as in section 13), or None."""
    frozen = grid_winner_name(emb_name)
    if frozen is None:
        return None
    names = [n for n in (frozen, frozen.replace("/linear", "-trainemb/linear")) if n in R]
    return max(names, key=lambda n: H.fold_scores(R[n]).mean()) if names else None


def final_names(emb_name, R):
    """(vanilla, centred) result names of the final systems; either is None if it hasn't been run."""
    v = best_vanilla_name(emb_name, R)
    c = v.replace("/linear", "-centred/linear") if v else None
    return v, (c if c in R else None)


def best_final_name(emb_name, R):
    """The better of the two final systems by CV mean, or None."""
    names = [n for n in final_names(emb_name, R) if n]
    return max(names, key=lambda n: H.fold_scores(R[n]).mean()) if names else None


def on_folds(res, folds):
    """`res` restricted to the given folds, to pair it with a run on fewer folds."""
    keep = {(r, k) for r, k, _, _ in folds}
    return {**res, "folds": [f for f in res["folds"] if (f["repeat"], f["fold"]) in keep]}


def rebuild(res):
    """(make_model, emb_name, train_kw) that retrain the system of a saved result as it was trained."""
    cfg = res["config"]
    E_ = emb_matrix(cfg["emb"])

    def make_model():
        return LSTMClassifier(
            E_.clone(),
            cfg["hidden"],
            cfg["layers"],
            cfg["dropout"],
            freeze=cfg.get("freeze", True),
            centre=cfg.get("centre", False),
        )

    return make_model, cfg["emb"], {k: v for k, v in cfg.items() if k not in MODEL_KEYS}


def rerun(res, name, folds=None, save=True, verbose=0, **kw):
    """Retrain the system of saved result `res` under a new name; kw goes to run_cv (seed, shuffle_test)."""
    make_model, emb_name, train_kw = rebuild(res)
    new = run_cv(make_model, name, emb_name, folds=folds, verbose=verbose, **kw, **train_kw)
    new["config"].update(
        {k: v for k, v in res["config"].items() if k in MODEL_KEYS and k != "seed"}
    )
    if save:
        H.save_result(new)
    return new

### 14.1 Grid report

The grid winner against every other config (A = winner, B = the config): Δ is B − A, and P(ROPE) is the probability that they are practically equivalent (within 1 pp). Then the mean balanced accuracy for each value of each hyperparameter, averaged over the other two (the grid is a full factorial, so these are balanced). Configs from a grid run before every config was saved show up as missing.

In [ ]:
def grid_report(emb_name):
    """(winner vs each config, mean per hyperparameter value) for the stored grid of `emb_name`."""
    grid = pd.read_csv(ROOT / "part2" / f"lstm_grid_{emb_name}.csv")
    R = H.load_results()
    name = lambda r: f"{emb_name}-lstm-h{int(r.hidden)}-l{int(r.layers)}-d{float(r.dropout)}/linear"
    winner = R[name(grid.iloc[0])]
    rows, cmps, missing = [], [], []
    for r in grid.iloc[1:].itertuples():
        if name(r) not in R:
            missing.append(name(r))
            continue
        c = H.compare(winner, R[name(r)])
        cmps.append(c)
        rows.append(
            {
                "config": f"h{int(r.hidden)} l{int(r.layers)} d{r.dropout}",
                "bal_acc": r.bal_acc,
                "diff_pp": 100 * c["mean_diff"],
                "P(winner better)": c["p_a_better"],
                "P(ROPE)": c["p_rope"],
                "P(config better)": c["p_b_better"],
            }
        )
    if missing:
        print(
            f"{emb_name}: {len(missing)} configs have no saved result; re-run run_pipeline('{emb_name}') to save them"
        )
    marginal = pd.concat(
        {
            hp: grid.groupby(hp)["bal_acc"].agg(["mean", "min", "max"])
            for hp in ("hidden", "layers", "dropout")
        }
    )
    if cmps:
        fig = A.plot_posteriors(cmps, ncols=4)
        print("saved", save_fig(fig, f"grid_posteriors_{emb_name}.png"))
        plt.show()
    return pd.DataFrame(rows), marginal

In [ ]:
for emb_name in ["ft-crawl", "glove-840b"]:
    if grid_winner_name(emb_name) is None:
        print(f"skip {emb_name}: run run_pipeline('{emb_name}') first")
        continue
    vs_winner, marginal = grid_report(emb_name)
    print(f"{emb_name}: winner {grid_winner_name(emb_name)}")
    display(vs_winner.round(4))
    display(marginal.round(4))

### 14.2 Selection optimism

The winner is the best of 12 configs on the same 5 folds that give its score, so part of its lead is luck: its CV score is biased upwards. The test split was not used for the choice, so the gap between the two estimates that bias. The test split has only 540 texts (30 prompts), so its own standard error is about 2 pp: a gap smaller than that is not evidence of optimism.

In [ ]:
def selection_optimism(emb_names=("ft-crawl", "glove-840b")):
    rows = []
    for emb_name in emb_names:
        csv = ROOT / "part2" / f"lstm_grid_{emb_name}.csv"
        test_path = ROOT / "part2" / f"lstm_test_{emb_name}.json"
        if not (csv.exists() and test_path.exists()):
            print(f"skip {emb_name}: run run_pipeline('{emb_name}') first")
            continue
        grid, test = pd.read_csv(csv), json.loads(test_path.read_text())
        acc = test["accuracy"]
        rows.append(
            {
                "emb": emb_name,
                "winner": test["name"],
                "CV winner": grid["bal_acc"].iloc[0],
                "CV runner-up": grid["bal_acc"].iloc[1],
                "CV grid median": grid["bal_acc"].median(),
                "test": test["balanced_accuracy"],
                "test SE (pp)": 100 * np.sqrt(acc * (1 - acc) / len(y_test)),
                "CV − test (pp)": 100 * (grid["bal_acc"].iloc[0] - test["balanced_accuracy"]),
            }
        )
    return pd.DataFrame(rows)


selection_optimism().round(4)

### 14.3 Seed variance

Each final system (best vanilla and centred, per embedding) is retrained with seeds 1, 2, 3 on the **same folds and inner splits**, so only the initialisation and the batch order change. Seed 0 is the saved run. The first table compares the spread across seeds (`seed_std`, std of the 5-fold mean) with the spread across folds (`fold_std`). The second repeats the vanilla-vs-centred comparison for each seed: if its decision changes with the seed, one run was not enough to make it. Results are saved as `<name>-seed<s>`.

In [ ]:
def seed_variance(emb_name, seeds=(1, 2, 3), folds=None, save=True, verbose=0):
    """(per-system summary, vanilla vs centred per seed) for the final systems of `emb_name`."""
    folds = FOLDS if folds is None else folds
    R = H.load_results()
    names = [n for n in final_names(emb_name, R) if n]
    if not names:
        print(f"skip {emb_name}: run sections 11-13 first")
        return None, None
    runs = {}  # (name, seed) -> result
    for n in names:
        runs[(n, 0)] = on_folds(R[n], folds)
        for s in seeds:
            runs[(n, s)] = rerun(
                R[n], n.replace("/linear", f"-seed{s}/linear"), folds, save, verbose, seed=s
            )
    per_run = pd.DataFrame(
        [
            {
                "system": n,
                "seed": s,
                "bal_acc": H.fold_scores(r).mean(),
                "fold_std": H.fold_scores(r).std(),
            }
            for (n, s), r in runs.items()
        ]
    )
    summary = per_run.groupby("system").agg(
        mean=("bal_acc", "mean"),
        seed_std=("bal_acc", "std"),
        fold_std=("fold_std", "mean"),
        n_seeds=("seed", "size"),
    )
    per_seed = None
    if len(names) == 2:  # vanilla vs centred, once per seed
        rows = []
        for s in (0, *seeds):
            c = H.compare(runs[(names[0], s)], runs[(names[1], s)])
            rows.append(
                {
                    "seed": s,
                    "centred − vanilla (pp)": 100 * c["mean_diff"],
                    "P(vanilla better)": c["p_a_better"],
                    "P(ROPE)": c["p_rope"],
                    "P(centred better)": c["p_b_better"],
                }
            )
        per_seed = pd.DataFrame(rows)

    fig, ax = plt.subplots(figsize=(1.5 + 1.8 * len(names), 3.4))
    for i, n in enumerate(names):
        v = per_run.loc[per_run["system"] == n, "bal_acc"].to_numpy()
        ax.scatter(
            i + np.linspace(-0.08, 0.08, len(v)),
            v,
            color=MUTED,
            s=30,
            zorder=2,
            label="one seed" if i == 0 else None,
        )
        ax.scatter(
            [i],
            [v.mean()],
            color=INK,
            s=90,
            marker="D",
            zorder=3,
            label="mean over seeds" if i == 0 else None,
        )
    ax.set_xticks(range(len(names)), ["centred" if "-centred" in n else "vanilla" for n in names])
    ax.set_xlim(-0.6, len(names) - 0.4)
    ax.set_ylabel("balanced accuracy (5-fold mean)")
    ax.set_title(f"{emb_name}: seed variance", fontsize=10)
    ax.grid(axis="y", color="#e6e6e3", lw=0.8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=9)
    print("saved", save_fig(fig, f"seed_variance_{emb_name}.png"))
    plt.show()
    return summary, per_seed

In [ ]:
SEED_RUNS = {}
for emb_name in ["ft-crawl", "glove-840b"]:
    summary, per_seed = seed_variance(emb_name)
    if summary is not None:
        SEED_RUNS[emb_name] = (summary, per_seed)
        display(summary.round(4))
        if per_seed is not None:
            display(per_seed.round(4))

### 14.4 More repeats: LSTM vs embed-and-pool

One repeat gives 5 folds, so the posterior of a difference is wide. The best final LSTM (vanilla or centred) is run on CV repeats 1 and 2 as well, and merged with its saved repeat 0 (runs are deterministic, so repeat 0 doesn't need to be redone): 15 folds, saved as `<name>-3rep`. It is then compared with the embed-and-pool probes on the same 15 folds (`H.compare` pairs on the shared repeats):
- `emb-ft-crawl-mean+func-promptcentred/logreg`: Liam's best embed-and-pool system (43.0%);
- `emb-<emb>-mean-promptcentred/logreg`: the plain prompt-centred mean of the same embedding.

A = embed-and-pool, B = LSTM.

In [ ]:
def more_repeats(emb_name, repeats=3, save=True, verbose=0):
    """(15-fold result, comparison table) for the best final LSTM of `emb_name` against embed-and-pool."""
    R = H.load_results()
    name = best_final_name(emb_name, R)
    if name is None:
        print(f"skip {emb_name}: run sections 11-13 first")
        return None, None
    base = {**R[name], "folds": [f for f in R[name]["folds"] if f["repeat"] == 0]}
    extra_folds = [f for f in H.load_folds(D.train["id"].tolist(), repeats=repeats) if f[0] > 0]
    extra = rerun(R[name], name, extra_folds, save=False, verbose=verbose)
    res = {
        **base,
        "name": name.replace("/linear", f"-{repeats}rep/linear"),
        "folds": base["folds"] + extra["folds"],
        "runs": repeats,
        "seconds": base["seconds"] + extra["seconds"],
    }
    if save:
        H.save_result(res)
    rows, cmps = [], []
    for b in dict.fromkeys(
        ["emb-ft-crawl-mean+func-promptcentred/logreg", f"emb-{emb_name}-mean-promptcentred/logreg"]
    ):
        if b not in R:
            print(f"no saved result for {b}")
            continue
        c = H.compare(R[b], res)
        cmps.append(c)
        pool = H.fold_scores(H.subset_repeats(R[b], repeats)).mean()
        rows.append(
            {
                "embed-and-pool": b,
                "pool": pool,
                "LSTM": H.fold_scores(res).mean(),
                "LSTM − pool (pp)": 100 * c["mean_diff"],
                "P(pool better)": c["p_a_better"],
                "P(ROPE)": c["p_rope"],
                "P(LSTM better)": c["p_b_better"],
                "folds": c["n_folds"],
            }
        )
    if cmps:
        fig = A.plot_posteriors(cmps, ncols=2)
        print("saved", save_fig(fig, f"lstm_vs_pool_{emb_name}.png"))
        plt.show()
    return res, pd.DataFrame(rows)

In [ ]:
REPEATS = {}
for emb_name in ["ft-crawl", "glove-840b"]:
    res, table = more_repeats(emb_name)
    if res is not None:
        REPEATS[emb_name] = res
        display(table.round(4))

## 15. Error analysis (answer 6)

Which characteristics of a text predict whether a system gets it right, and where do the major design choices differ? Five systems are compared on the same texts, using each one's out-of-fold predictions (CV repeat 0: every train text is predicted exactly once, by a model that never saw its prompt):

| label | system | design choice it represents |
|---|---|---|
| pool raw | `emb-ft-crawl-mean/logreg` | embed-and-pool, no centring |
| pool centred | `emb-ft-crawl-mean+func-promptcentred/logreg` | best embed-and-pool (Liam) |
| LSTM vanilla | best vanilla ft-crawl LSTM (section 12) | recurrent encoder |
| LSTM centred | its prompt-centred version (section 13) | recurrent encoder + centring |
| LSTM GloVe | best GloVe LSTM | embedding choice |

Steps:
5. **Analysis table**: one row per text, with analysis features and a correct/wrong column per system.
6. **Bayesian regression of correctness on the features** (the course's BDA model, `part2/analysis.py`), with the true author as a control; plus accuracy by length.
7. **Per author**: which models each design choice helps or hurts.
8. **Word-order ablation**: does the LSTM use word order at all?
9. **Minimal pairs**: does the LSTM react to negation and to the position of a connective?

In [ ]:
import textstat
from nltk.corpus import stopwords
from scipy.spatial.distance import jensenshannon

from part1 import eda as EDA
from part2 import analysis as BDA

NEGATION = r"\b(?:not|no|never|nothing|nor|none|nobody|neither|nowhere|cannot)\b|n't"  # list from the answers doc
STOPWORDS = set(stopwords.words("english"))
ANALYSIS_FEATURES = [
    "n_words",
    "letters_per_word",
    "comma_per100",
    "semicolon_per100",
    "colon_per100",
    "em_dash_per100",
    "contractions_per100",
    "negation_per100",
    "curly_share",
    "func_share",
    "ttr",
    "unk_share",
    "flesch",
]
PAIRS = [
    ("pool raw", "LSTM vanilla"),
    ("pool centred", "LSTM centred"),
    ("LSTM vanilla", "LSTM centred"),
]
SYS_COLORS = [
    "#2a78d6",
    "#eb6834",
    "#1baf7a",
    "#eda100",
    "#e87ba4",
]  # categorical slots 1-5, in this fixed order


def analysis_systems(R):
    """{label: result name} of the systems in the table above that have a saved result."""
    ft_vanilla, ft_centred = final_names("ft-crawl", R)
    systems = {
        "pool raw": "emb-ft-crawl-mean/logreg",
        "pool centred": "emb-ft-crawl-mean+func-promptcentred/logreg",
        "LSTM vanilla": ft_vanilla,
        "LSTM centred": ft_centred,
        "LSTM GloVe": best_final_name("glove-840b", R),
    }
    missing = [k for k, v in systems.items() if v not in R]
    if missing:
        print("no saved result, left out:", missing)
    return {k: v for k, v in systems.items() if v in R}

### 15.1 Analysis table

Features, all per text:
- **length and words:** `n_words` (whitespace words), `letters_per_word`, `ttr` (type-token ratio of the lowercased words), `func_share` (share of words that are nltk stopwords), `flesch` (Flesch reading ease, textstat);
- **punctuation** (per 100 words, `part1/eda.py`): comma, semicolon, colon, em dash;
- **habits** (per 100 words): contractions (`part1/eda.py`), negations (the list in the answers doc);
- **quotes:** `curly_share` = curly / all quote characters. Texts with no quotes get the mean, so they don't count as "straight";
- **coverage:** `unk_share` = share of fastText tokens without a vector.

Plus `ok:<system>` = 1 if that system predicted the text's author correctly.

In [ ]:
def analysis_table(R=None):
    R = H.load_results() if R is None else R
    df = EDA.add_length_features(D.train[["text"]].reset_index(drop=True))
    df = EDA.add_vocab_features(EDA.add_punctuation_features(df))
    low = df["text"].str.lower().str.replace("’", "'", regex=False)
    df["negation_per100"] = low.str.count(NEGATION) / df["n_words"] * 100
    curly = df["n_curly_apos"] + df["n_curly_dquote"]
    quotes = curly + df["n_straight_apos"] + df["n_straight_dquote"]
    df["curly_share"] = (curly / quotes.replace(0, np.nan)).fillna(
        (curly / quotes.replace(0, np.nan)).mean()
    )
    words = low.str.findall(r"[a-z']+")
    df["func_share"] = words.apply(lambda w: np.mean([x in STOPWORDS for x in w]) if w else 0.0)
    df["ttr"] = words.apply(lambda w: len(set(w)) / len(w) if w else 0.0)
    df["unk_share"] = [np.mean(np.asarray(s) == UNK_IDX) for s in IDS["ft-crawl"]]
    df["flesch"] = df["text"].apply(textstat.flesch_reading_ease)
    df["y"], df["author"], df["prompt"] = y, [CLASSES[i] for i in y], groups
    for label, n in analysis_systems(R).items():
        df[f"ok:{label}"] = (np.asarray(R[n]["oof_pred"]) == y).astype(int)
    assert df[ANALYSIS_FEATURES].notna().all().all()
    return df


TABLE = analysis_table()
display(TABLE[ANALYSIS_FEATURES].describe().T.round(3))
TABLE.filter(like="ok:").mean().rename("accuracy").to_frame().round(4)

### 15.2 What predicts correctness: BDA model

The course's Bayesian model: for each system, the logit of "correct" is a system bias plus a weighted sum of the standardised features, with Laplace(0, 1) priors that pull weights to 0 unless the data say otherwise. The true author's one-hot is added as a **control** (as the tutorial adds the predicted class): some authors are both easier and, say, heavier em-dash users, so without it a feature weight would partly be an author effect. With it, a weight means "**within an author**, texts with more of this feature are easier (+) or harder (−) for this system".

- **Forest plot:** the median and 94% interval of each feature weight, per system.
- **Table:** for each design choice (pair of systems), the probability that the second system's weight is larger. Values near 0 or 1 mark the features where the two systems **differ** in what makes a text hard.
- **Accuracy by length:** plain accuracy per quartile of `n_words`, one line per system.

MCMC with 2 chains × 500 draws takes a few minutes.

In [ ]:
def bda_analysis(table, features=ANALYSIS_FEATURES, num_warmup=500, num_samples=500, num_chains=2):
    """(posterior, P(w_B > w_A) per feature for PAIRS); also plots the weights and accuracy by length."""
    systems = [c[3:] for c in table.columns if c.startswith("ok:")]
    n = len(table)
    X = np.hstack(
        [table[list(features)].to_numpy(float), BDA.one_hot(table["y"].to_numpy(), N_CLASSES)]
    )
    mcmc = BDA.fit_bda_model(
        len(systems),
        np.repeat(
            np.arange(len(systems)), n
        ),  # system id of each row: systems stacked one after the other
        np.tile(X, (len(systems), 1)),
        np.concatenate([table[f"ok:{s}"].to_numpy() for s in systems]),
        num_warmup=num_warmup,
        num_samples=num_samples,
        num_chains=num_chains,
    )
    post = BDA.posterior_to_arviz(mcmc, systems, list(features) + [f"author:{c}" for c in CLASSES])
    w = post["posterior"]["w"].transpose("chain", "draw", "system", "feature").values
    w = w.reshape(-1, len(systems), X.shape[1])[:, :, : len(features)]  # draws x systems x features

    fig, ax = plt.subplots(figsize=(8, 0.55 * len(features) + 1.2))
    offsets = np.linspace(-0.3, 0.3, len(systems))
    for j, s in enumerate(systems):
        lo, mid, hi = np.percentile(w[:, j, :], [3, 50, 97], axis=0)
        rows_y = np.arange(len(features)) + offsets[j]
        ax.hlines(rows_y, lo, hi, color=SYS_COLORS[j], lw=2)
        ax.plot(mid, rows_y, "o", color=SYS_COLORS[j], ms=4, label=s)
    ax.axvline(0, color=MUTED, lw=1)
    ax.set_yticks(range(len(features)), features)
    ax.invert_yaxis()
    ax.set_xlabel("weight on the logit of correct, per SD of the feature (median, 94% interval)")
    ax.grid(axis="x", color="#e6e6e3", lw=0.8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=9, loc="upper left", bbox_to_anchor=(1.01, 1))
    ax.set_title("What makes a text easy (+) or hard (−), within an author", fontsize=10)
    print("saved", save_fig(fig, "bda_weights.png"))
    plt.show()

    idx = {s: j for j, s in enumerate(systems)}
    pairs = {
        f"P(w[{b}] > w[{a}])": (w[:, idx[b], :] > w[:, idx[a], :]).mean(0)
        for a, b in PAIRS
        if a in idx and b in idx
    }

    acc = table.groupby(pd.qcut(table["n_words"], 4), observed=True)[
        [f"ok:{s}" for s in systems]
    ].mean()
    fig, ax = plt.subplots(figsize=(6.5, 3.6))
    for j, s in enumerate(systems):
        ax.plot(
            range(len(acc)), acc[f"ok:{s}"], marker="o", ms=6, lw=2, color=SYS_COLORS[j], label=s
        )
    ax.set_xticks(range(len(acc)), [f"{int(i.left) + 1}–{int(i.right)}" for i in acc.index])
    ax.set_xlabel("words per text (quartiles)")
    ax.set_ylabel("accuracy")
    ax.grid(axis="y", color="#e6e6e3", lw=0.8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=9, loc="upper left", bbox_to_anchor=(1.01, 1))
    print("saved", save_fig(fig, "accuracy_by_length.png"))
    plt.show()
    return post, pd.DataFrame(pairs, index=list(features))

In [ ]:
POST, weight_pairs = bda_analysis(TABLE)
weight_pairs.round(3)

### 15.3 Per author

Recall per author for each system, then, for each design choice, the change in recall per author (second system − first, in pp). Balanced accuracy is the mean of these recalls, so this shows **which authors** a mean gain comes from. Then the confusion matrix of the best LSTM (rows and columns ordered by family).

In [ ]:
def per_author(R=None):
    """(recall per author and system, recall difference per author for PAIRS); plots both and a confusion matrix."""
    R = H.load_results() if R is None else R
    systems = analysis_systems(R)
    recall = pd.DataFrame(
        {
            label: A.per_class_f1(y, np.asarray(R[n]["oof_pred"]), D)["recall"]
            for label, n in systems.items()
        }
    )
    diffs = pd.DataFrame(
        {f"{b} − {a}": recall[b] - recall[a] for a, b in PAIRS if a in recall and b in recall}
    )

    fig, axes = plt.subplots(
        1, diffs.shape[1], figsize=(3.4 * diffs.shape[1], 5.5), sharey=True, squeeze=False
    )
    for ax, col in zip(axes[0], diffs.columns):
        d = 100 * diffs[col].to_numpy()
        ax.axvline(0, color=MUTED, lw=1)
        ax.hlines(range(len(d)), 0, d, color=INK, lw=1)
        ax.plot(d, range(len(d)), "o", color=INK, ms=5)
        ax.set_title(f"{col}\nmean {d.mean():+.1f} pp", fontsize=10)
        ax.set_xlabel("recall difference (pp)")
        ax.grid(axis="x", color="#e6e6e3", lw=0.8)
        ax.spines[["top", "right"]].set_visible(False)
    axes[0, 0].set_yticks(range(len(diffs)), diffs.index)
    axes[0, 0].invert_yaxis()
    fig.tight_layout()
    print("saved", save_fig(fig, "per_author_diff.png"))
    plt.show()

    lstms = [l for l in systems if l.startswith("LSTM")]
    if lstms:
        best = max(lstms, key=lambda l: H.fold_scores(R[systems[l]]).mean())
        GRAPHS.mkdir(parents=True, exist_ok=True)
        path = GRAPHS / f"confusion_{systems[best].replace('/', '__')}.png"
        A.plot_confusion(
            y, np.asarray(R[systems[best]]["oof_pred"]), D, title=systems[best], path=path
        )
        print("saved", path)
        plt.show()
    return recall, diffs

In [ ]:
RECALL, recall_diffs = per_author()
display(RECALL.round(3))
(100 * recall_diffs).round(1)

### 15.4 Word-order ablation

Mean pooling ignores word order by construction. Does the LSTM use it? The best final ft-crawl and GloVe LSTMs are retrained on the same folds (so the original scores equal the saved ones), and each fold's test texts are also scored with **their tokens randomly shuffled**: same tokens, same counts, no order. If the shuffled score is about the same, the LSTM is working as a (learned) bag of embeddings, which would explain why it doesn't beat embed-and-pool. A = original, B = shuffled. About one CV run per embedding; saved as `<name>-ordertest`.

In [ ]:
def order_ablation(emb_name, folds=None, save=True, verbose=0):
    """(per-fold table, comparison shuffled vs original) for the best final LSTM of `emb_name`."""
    R = H.load_results()
    name = best_final_name(emb_name, R)
    if name is None:
        print(f"skip {emb_name}: run sections 11-13 first")
        return None, None
    res = rerun(
        R[name],
        name.replace("/linear", "-ordertest/linear"),
        folds,
        save,
        verbose,
        shuffle_test=True,
    )
    shuffled = {
        **res,
        "name": res["name"] + " (shuffled)",
        "folds": [
            {**f, "balanced_accuracy": f["balanced_accuracy_shuffled"]} for f in res["folds"]
        ],
    }
    c = H.compare(res, shuffled)
    table = pd.DataFrame(
        [
            {
                "fold": f["fold"],
                "original": f["balanced_accuracy"],
                "shuffled": f["balanced_accuracy_shuffled"],
            }
            for f in res["folds"]
        ]
    )
    print(
        f"{name}: shuffled − original {100 * c['mean_diff']:+.1f} pp, P(original better) {c['p_a_better']:.3f}, "
        f"P(ROPE) {c['p_rope']:.3f}, P(shuffled better) {c['p_b_better']:.3f}"
    )
    return table, c

In [ ]:
ORDER = {}
for emb_name in ["ft-crawl", "glove-840b"]:
    table, c = order_ablation(emb_name)
    if table is not None:
        ORDER[emb_name] = c
        display(table.round(4))

### 15.5 Minimal pairs

One model of the best final ft-crawl system is trained on the whole train split (early stopping on 20% of the prompts, as in `final_test`). Each pair of sentences is then scored, and the Jensen–Shannon divergence between the two predicted author distributions measures how differently the model treats them (0 = identical, 1 = no overlap). Mean pooling gives exactly 0 for the reordered pairs, since the tokens are the same. The last pair (unrelated sentences) is the scale for a "large" difference.

Caveats:
- single sentences are much shorter than our ~120-word passages, so this is qualitative;
- a centred model sees each pair as a batch of 2 and subtracts the pair's mean, so it can only see the difference between the two sentences, which is what we want to measure here anyway.

In [ ]:
MINIMAL_PAIRS = [
    ("negation", "The result is good.", "The result is not good."),
    ("connective position", "Hence, it works.", "It works, hence."),
    ("contraction", "I do not know.", "I don't know."),
    ("word order", "The old man saw the dog.", "The dog saw the old man."),
    ("unrelated (scale)", "The result is good.", "Birds migrate south in winter."),
]


def minimal_pairs(emb_name="ft-crawl", pairs=MINIMAL_PAIRS, verbose=False):
    R = H.load_results()
    name = best_final_name(emb_name, R)
    if name is None:
        print(f"skip {emb_name}: run sections 11-13 first")
        return None
    make_model, emb_name, train_kw = rebuild(R[name])
    H.set_seed(H.SEED)
    itr, iva = H.inner_split(np.arange(len(y)), groups, seed=H.SEED)
    model, _, _ = train_fold(make_model, itr, iva, emb_name, verbose=verbose, **train_kw)
    model.eval()
    tok_ = EMB[emb_name][0]
    rows = []
    for kind, a, b in pairs:
        seqs = [torch.tensor(tok_.ids(s), dtype=torch.long) for s in (a, b)]
        x = pad_sequence(seqs, batch_first=True, padding_value=PAD_IDX)
        with torch.no_grad():
            p = (
                torch.softmax(model(x, torch.tensor([len(s) for s in seqs])), -1)
                .numpy()
                .astype(float)
            )
        rows.append(
            {
                "pair": kind,
                "A": a,
                "B": b,
                "top A": CLASSES[p[0].argmax()],
                "top B": CLASSES[p[1].argmax()],
                "JS divergence": jensenshannon(p[0], p[1], base=2) ** 2,
            }
        )
    print(f"model: {name}, trained on the whole train split")
    return pd.DataFrame(rows)


minimal_pairs()